In [1]:
import polars as pl
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from pathlib import Path

FEATURES_DIR = Path("../data/processed/features_v5")

def rmsle_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(np.clip(y_true, 0, None)),
                                      np.log1p(np.clip(y_pred, 0, None))))

def load_fold(fold_path: Path) -> pd.DataFrame:
    return pl.read_parquet(fold_path / "batch_*.parquet").to_pandas()

print("Загрузка фолдов v5 для Optuna...")
train_dfs = [load_fold(FEATURES_DIR / f"fold_{i:02d}") for i in range(4)]
train_df = pd.concat(train_dfs, ignore_index=True)
val_df = load_fold(FEATURES_DIR / "fold_04")

drop_cols = ["user_id", "anchor_date", "target"]
features = [c for c in val_df.columns if c not in drop_cols]

print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,} | Features: {len(features)}")

X_train = train_df[features]
y_train_log = np.log1p(np.clip(train_df["target"].values, 0, None))

X_val = val_df[features]
y_val = val_df["target"].values
y_val_log = np.log1p(np.clip(y_val, 0, None))

train_pool = Pool(X_train, y_train_log)
val_pool = Pool(X_val, y_val_log)

Загрузка фолдов v5 для Optuna...
Train size: 1,000,000 | Val size: 250,000 | Features: 435


In [2]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 2000, 4000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.06, log=True),
        'depth': trial.suggest_int('depth', 6, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 15.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': 42,
        'od_type': 'Iter',
        'early_stopping_rounds': 150,
        'verbose': False,
    }
    
    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool)
    
    val_pred = np.expm1(np.clip(model.predict(X_val), 0, None))
    score = rmsle_score(y_val, val_pred)
    
    return score

In [3]:
N_OPTUNA_TRIALS = 25

def print_callback(study, trial):
    status = "PRUNED" if trial.state == optuna.trial.TrialState.PRUNED else f"RMSLE={trial.value:.5f}"
    print(f"  Trial {trial.number+1:2d}/{N_OPTUNA_TRIALS}: {status} "
          f"(best: {study.best_value:.5f}, trial #{study.best_trial.number+1})")

print(f"=== Запуск Optuna Tuning ({N_OPTUNA_TRIALS} итераций) ===")
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
optuna.logging.set_verbosity(optuna.logging.WARNING)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[print_callback])

print(f"\nЛучший RMSLE на валидации: {study.best_value:.5f}")
print("Идеальные гиперпараметры:")
for k, v in study.best_params.items():
    print(f"  '{k}': {v},")

[I 2026-08-23 20:30:21,136] A new study created in memory with name: no-name-7eb26de0-322b-4c82-97d1-5fbf57af8d36


=== Запуск Optuna Tuning (25 итераций) ===
  Trial  1/25: RMSLE=1.65945 (best: 1.65945, trial #1)
  Trial  2/25: RMSLE=1.66573 (best: 1.65945, trial #1)
  Trial  3/25: RMSLE=1.66690 (best: 1.65945, trial #1)
  Trial  4/25: RMSLE=1.66410 (best: 1.65945, trial #1)
  Trial  5/25: RMSLE=1.66187 (best: 1.65945, trial #1)
  Trial  6/25: RMSLE=1.65634 (best: 1.65634, trial #6)
  Trial  7/25: RMSLE=1.67012 (best: 1.65634, trial #6)
  Trial  8/25: RMSLE=1.66797 (best: 1.65634, trial #6)
  Trial  9/25: RMSLE=1.66634 (best: 1.65634, trial #6)
  Trial 10/25: RMSLE=1.65899 (best: 1.65634, trial #6)
  Trial 11/25: RMSLE=1.66689 (best: 1.65634, trial #6)
  Trial 12/25: RMSLE=1.65938 (best: 1.65634, trial #6)
  Trial 13/25: RMSLE=1.65959 (best: 1.65634, trial #6)
  Trial 14/25: RMSLE=1.66198 (best: 1.65634, trial #6)
  Trial 15/25: RMSLE=1.65623 (best: 1.65623, trial #15)
  Trial 16/25: RMSLE=1.66308 (best: 1.65623, trial #15)
  Trial 17/25: RMSLE=1.65693 (best: 1.65623, trial #15)
  Trial 18/25: RMSL